# RunPod · 판정 모델 (답이 갈린 문항 교차 검증) — `MODEL_ID`만 바꿔서 실행 · 학습 없음

**하는 일**
1. Colab `anchor_rescue` 6-1번 셀이 HF 비공개 저장소에 올린 **판정 목록**을 받습니다.
   - 판정 목록은 기준 모델(Flash-Next)과 보조 모델의 답이 갈린 문항입니다(검증·test).
2. 문항마다 이미지·질문·보기와 함께 두 답을 "모델 A의 답 / 모델 B의 답"으로 보여 줍니다. 어느 쪽이 기준인지는 문항마다 무작위입니다.
   - 두 후보 답(모델 A/B)을 보여 주고 근거를 교차 검증하게 하는 프롬프트로 묻습니다. 흔한 착각 패턴을 배제하고 단계별로 결론을 내게 합니다.
   - 마지막 줄 `정답: (x)`를 읽습니다.
3. 문항마다 표본 `K_SAMPLES`개(기본 5개)를 뽑고, 다수결로 이 판정 모델의 **한 표**를 정합니다. 표본이 모자라거나 최다 표가 동점이면 기권합니다.
4. 결과를 E14 형식으로 저장하고 HF 저장소 `vqa_runs/xc_<모델>_up20`에 올립니다. 목록 밖 문항은 기준 답으로 채웁니다(표를 내지 않는다는 뜻).
5. 브리지 B 단계를 거쳐 `anchor_rescue`를 다시 돌리면 판정 모델이 한 표씩 들어갑니다. 뒤집을지는 같은 검증(교차 적합·부트스트랩·2배 법칙)이 정합니다.

**실행 순서**
1. B300 pod Jupyter에서 **Run All**
   - 설치·데이터·가중치 캐시는 zero-shot 노트북과 같은 곳을 씁니다. 이미 받은 것은 건너뜁니다.
2. 다음 판정 모델: 1번 셀 `MODEL_ID`만 바꾸고 **Kernel → Restart Kernel and Run All Cells**
   - 추천 순서: `google/gemma-4-31B-it` → `Qwen/Qwen3.5-122B-A10B-GPTQ-Int4` → `naver-hyperclovax/HyperCLOVAX-SEED-Think-32B`
3. 다 끝나면 Colab 브리지 B 단계 → `anchor_rescue` 다시 실행 → pod **Terminate**

**사전 기준 (결과를 보기 전에 고정)**
- 판정 모델은 모두 같은 프롬프트·표본 5개·T 0.7·생각 모드 끔으로 돌립니다.
- 돌린 판정 모델은 성적과 상관없이 모두 넣습니다. 성적을 보고 골라 빼면 과적합입니다.
- 채택 여부는 `anchor_rescue`의 검증이 정합니다.

끊기면 다시 Run All 하면 이어서 합니다. B300 오류 대책(Triton 어텐션·Triton MoE·FlashInfer 자동 튜닝 끔)은 zero-shot 노트북과 같습니다.


## 1. 설정 — `MODEL_ID`만 바꾸세요


In [ ]:
MODEL_ID  = "naver-hyperclovax/HyperCLOVAX-SEED-Think-32B"   # ← 판정 모델 (이 파일은 HyperCLOVA X용으로 맞춰 둠). 여기만 바꾸세요. 추천 순서: google/gemma-4-31B-it → Qwen/Qwen3.5-122B-A10B-GPTQ-Int4 → naver-hyperclovax/HyperCLOVAX-SEED-Think-32B
PRESET    = "up2.0"                   # 해상도 (다른 멤버와 같음)
RUN_NAME  = "auto"                    # "auto" → xc_<모델 이름>_<해상도>
TARGETS_FILE = ""                     # 판정 목록. 비워 두면 HF 비공개 저장소의 xcheck/xcheck_targets.csv (Colab anchor_rescue 6-1번 셀이 올림). 직접 올렸으면 그 파일 경로

# ── 교차 검증 프롬프트 생성 (결과를 보기 전에 고정 · 판정 모델 모두 같은 값) ──
K_SAMPLES   = 5                       # 문항당 표본 수 → 다수결 = 이 판정 모델의 한 표
TEMPERATURE, TOP_P, TOP_K = 0.7, 0.8, 20
MAX_TOKENS  = 1200                    # 단계별 풀이 + 마지막 줄 '정답: (x)'. 첫 묶음 점검에서 '길이 제한에 걸림'이 많으면 늘림
MIN_VALID   = 3                       # 답을 읽어 낸 표본이 이보다 적거나 최다 표가 동점이면 기권 (= 표를 내지 않음)
THINKING    = False                   # 생각 모드 끔 (템플릿이 지원할 때만 적용)
GEN_CHUNK   = 32                      # 한 번에 넘기는 문항 수 = 저장 단위
GPU_ID    = None                      # GPU 여러 장 pod에서 노트북 여러 개를 동시에: 노트북마다 0, 1, 2, … (None = 전부 보이게)
VAL_N, PROMPT_VERSION_USE, ANSWER_PREFIX = 1000, "baseline_v1", ""   # 아래 공용 셀용 (바꾸지 마세요)

# ── 모델별 설정 (여기 없는 모델은 아래 공통 기본값) ──
MODEL_PRESETS = {
    "google/gemma-4-31B-it":            {},                                   # 판정 1순위: zero-shot 97.67% (앞 300문항) · Qwen과 다른 계열. 라이선스 동의 + HF 토큰
    "Qwen/Qwen3.5-122B-A10B-GPTQ-Int4": {"QUANTIZATION": "moe_wna16"},       # 판정 2순위 (zero-shot 검증 96.0%)
    "naver-hyperclovax/HyperCLOVAX-SEED-Think-32B": {"TRUST_REMOTE_CODE": True, "REVISION": "a6cdfd3464d1b767259cad23e164eaf39d3e3960"},   # 판정 3순위: 네이버·한국어 (95.67%). HF main은 transformers 5.18+ 전용 형식 → vLLM 0.30이 시험한 원격 코드판 커밋으로 고정 · 생각 블록은 5번 셀이 닫음
    "OpenGVLab/InternVL3_5-38B-HF":     {},                                   # 선택 (94.33%)
    "Qwen/Qwen3.5-27B":                 {},                                   # 선택 (96.33%, 122B와 같은 계열)
    "Qwen/Qwen3.6-35B-A3B":             {},                                   # 선택 (96.33%, 122B와 같은 계열)
}
# ── 공통 기본값 (vLLM) ──
DTYPE         = "auto"          # 체크포인트 dtype 그대로 (bf16)
QUANTIZATION  = None            # 양자화 판은 MODEL_PRESETS에서 정함
TRUST_REMOTE_CODE = False
REVISION      = None            # HF 저장소 커밋 고정 (None = main). HyperCLOVA X는 MODEL_PRESETS가 고정
MOE_BACKEND   = "triton"        # B300: FlashInfer MoE 커널은 즉석 컴파일이 필요한데 pod의 nvcc가 B300(sm_103)을 몰라 실패 → Triton (MoE 모델만 해당)
ATTN_BACKEND  = "TRITON_ATTN"   # B300: 같은 이유로 FlashInfer 어텐션 대신 Triton 어텐션. None = vLLM 자동 선택(G4·H100 등)
GPU_MEM_UTIL  = 0.92
MAX_MODEL_LEN = 8192
MAX_NUM_SEQS  = 64              # 생성은 표본 여러 개를 동시에 돌려야 빠름 (시작이 실패하면 32)
MAX_BATCHED_TOKENS = 32768
TOPK_LOGPROBS = 64
CHUNK         = 128             # 한 번에 넘기는 문항 수 = 저장 단위
IMG_WORKERS   = 8
GDN_PREFILL_BACKEND = "triton"  # Qwen3.5 계열 선형 어텐션 커널 (다른 모델에는 영향 없음)
ENGINE_KW     = {}
for _k, _v in MODEL_PRESETS.get(MODEL_ID, {}).items(): globals()[_k] = _v
if ATTN_BACKEND: ENGINE_KW.setdefault("attention_config", {"backend": ATTN_BACKEND})
ENGINE_KW.setdefault("enable_flashinfer_autotune", False)   # FlashInfer 자동 튜닝(즉석 컴파일)도 건너뜀 → "nvcc fatal: Unsupported gpu architecture 'compute_103a'" 방지

import re as _re
if RUN_NAME == "auto":
    RUN_NAME = "xc_" + _re.sub(r"[^a-z0-9]+", "_", MODEL_ID.split("/")[-1].lower()).strip("_") + "_" + PRESET.replace(".", "") + ("" if PROMPT_VERSION_USE == "baseline_v1" else "_" + PROMPT_VERSION_USE)
VAL_ORDER_SEED = 2026           # 앞 1,000(팀 공통 검증) 뒤의 train 5,714문항 순서 (고정)

# ── RunPod 경로 (바꾸지 마세요. 모두 볼륨 디스크 /workspace 안) ──
ZIP_PATH     = "/workspace/ssafy-16-2-ai.zip"   # 대회 데이터 zip. 없으면 3번 셀이 HF 비공개 저장소에서 받음
ZIP_PASSWORD = "YOUR_ZIP_PASSWORD"
DATA_DIR     = "/workspace/data"
OUT_DIR      = "/workspace/vqa_runs"
HF_CACHE     = "/workspace/hf"                  # 가중치 캐시는 볼륨 디스크의 이 한 곳 (Flash-Next 노트북·터미널 다운로드와 같은 곳)
HF_REPO      = ""                               # 비워 두면 "<내 HF 아이디>/ssafy-vqa-private" (Colab 브리지 노트북과 같은 규칙)
HF_ZIP_NAME  = "ssafy-16-2-ai.zip"
SEED, VAL_SIZE = 42, 1000
HF_TOKEN   = ""                 # 비워 두면 pod 환경 변수 HF_TOKEN → 그것도 없으면 3번 셀에서 입력창 (Write 권한 토큰. Gemma는 라이선스 동의한 계정)

RES_PRESETS = {
    "baseline384": (384 * 384, 384 * 384),
    "mid":         (256 * 28 * 28, 512 * 28 * 28),
    "native":      (256 * 28 * 28, 1280 * 28 * 28),
    "up1.5":       (1_550_000, 2_100_000),
    "up2.0":       (2_700_000, 3_400_000),
}
RES_PRESETS_GEMMA4 = {"baseline384": 70, "mid": 140, "native": 280, "up1.5": 560, "up2.0": 1120}   # stage2와 같음 (Gemma 4는 이미지당 토큰 수)
print(f"MODEL_ID = {MODEL_ID} | RUN_NAME = {RUN_NAME} | PRESET = {PRESET} | 표본 {K_SAMPLES}개 · T {TEMPERATURE} · 최대 {MAX_TOKENS}토큰 | 설정 덮어쓰기 {MODEL_PRESETS.get(MODEL_ID, {}) or '없음 (기본값)'}")

import os, sys
# vLLM이 같이 까는 torchaudio(CUDA 12.8용)가 torch 2.13(CUDA 13)과 섞이면 import 순간 RuntimeError가 남 (2번 셀이 지우지만, 셀 순서가 꼬여도 안전하게)
# (transformers의 AutoProcessor와 vLLM이 torchaudio가 '있으면' 불러옴). 이 노트북은 오디오를 쓰지 않으므로
# 이 프로세스에서는 torchaudio를 '없는 것'으로 취급 → 둘 다 torchaudio 없이 동작. 반드시 transformers·vLLM보다 먼저 실행되는 이 셀에 둠
sys.modules["torchaudio"] = None
if GPU_ID is not None:
    assert "torch" not in sys.modules, "GPU_ID는 커널을 새로 시작한 직후에만 적용됩니다 → Kernel → Restart Kernel and Run All Cells"
    os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID); print(f"이 노트북은 GPU {GPU_ID}번만 씁니다")
_HF_PRE = {k: os.environ[k] for k in ("HF_HOME", "HF_HUB_CACHE", "HUGGINGFACE_HUB_CACHE", "HF_XET_CACHE") if os.environ.get(k) and not os.environ[k].startswith(HF_CACHE)}
os.environ.update({"HF_HOME": HF_CACHE, "HF_HUB_CACHE": HF_CACHE + "/hub", "HF_XET_CACHE": HF_CACHE + "/xet"}); os.environ.pop("HUGGINGFACE_HUB_CACHE", None)   # pod에 미리 정해진 캐시 경로를 덮어써 두 번 받지 않게
if _HF_PRE: print(f"(pod에 미리 정해진 HF 캐시 경로 {_HF_PRE} 대신 {HF_CACHE} 하나만 씁니다)")
assert os.path.isdir("/workspace"), "/workspace가 없습니다 → pod 설정에서 Volume Disk(마운트 경로 /workspace)를 켰는지 확인하세요"
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")             # 대용량 다운로드 가속
os.environ.setdefault("VLLM_ENABLE_V1_MULTIPROCESSING", "0")      # vLLM 엔진을 노트북 프로세스 안에서 실행 (Colab의 fork/spawn 문제 회피)
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")         # 초기화 때 top-k/top-p 표본 추출을 FlashInfer(JIT 컴파일 필요)로 시험하지 않게 → PyTorch판. 이 노트북은 그리디 1토큰이라 무관


## 2. 설치 (zero-shot 노트북과 같음, 이미 설치돼 있으면 건너뜀)


In [ ]:
import subprocess, re, shutil, sys
from importlib.metadata import version as _ver, PackageNotFoundError
VLLM_VERSION = "0.30.0"
def _installed(pkg):
    try: return _ver(pkg)
    except PackageNotFoundError: return None
_smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(_smi[:800])
_m = re.search(r"CUDA Version:\s*([\d.]+)", _smi); DRIVER_CUDA = float(_m.group(1)) if _m else 0.0
if _installed("vllm") == VLLM_VERSION and _installed("transformers") == "5.17.0":
    print(f"vLLM {VLLM_VERSION} 이미 설치됨 → 건너뜀")
elif DRIVER_CUDA >= 13.0:
    !pip -q install "vllm=={VLLM_VERSION}" "transformers==5.17.0"
else:
    print(f"드라이버가 지원하는 CUDA {DRIVER_CUDA} < 13.0 → CUDA 12.9용 vLLM 휠을 받습니다")
    !pip -q install "https://github.com/vllm-project/vllm/releases/download/v{VLLM_VERSION}/vllm-{VLLM_VERSION}+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" "transformers==5.17.0" --extra-index-url https://download.pytorch.org/whl/cu129
print("vllm", _installed("vllm"), "| transformers", _installed("transformers"), "| torch", _installed("torch"))
# vLLM 0.30은 의존성으로 torchaudio==2.11.0(PyPI판 = CUDA 12.8용)을 같이 깔고, 이것이 torch 2.13(CUDA 13)과 섞이면 import 순간 RuntimeError가 남.
# 1번 셀의 차단은 이 노트북 프로세스에만 닿고, vLLM은 모델 구조를 '별도 프로세스'에서 검사하므로 거기서는 막을 수 없음 → 아예 지움 (오디오 미사용)
# RunPod 이미지에는 pandas가 없고, cutlass 모듈이 빠져 vLLM이 멈추는 경우가 있음 (실제 B300 pod에서 확인) → 없을 때만 설치
_extra = [p for p in ("pandas", "nvidia-cutlass") if not _installed(p)]
if _extra:
    _extra_s = " ".join(_extra)
    !pip -q install {_extra_s}
    print("추가 설치:", _extra)
if _installed("torchaudio"):
    !pip -q uninstall -y torchaudio
    print("torchaudio 제거 (vLLM이 같이 깐 CUDA 12.8용 · 오디오 미사용)")
# 설치 점검: vLLM의 모델 검사 프로세스와 같은 조건(차단 없는 새 프로세스)에서 Qwen3.5 모델 모듈과 AutoProcessor를 import해 봄
_chk = subprocess.run([sys.executable, "-c", "import torch, torchvision, transformers, vllm; from transformers import AutoProcessor; "
                       "import vllm.model_executor.models.qwen3_5; "
                       "print('CHECK', torch.__version__, torch.version.cuda, torchvision.__version__, transformers.__version__, vllm.__version__)"],
                      capture_output=True, text=True)
_line = [l for l in _chk.stdout.splitlines() if l.startswith("CHECK")]
if _chk.returncode != 0 or not _line:
    print(_chk.stdout[-1500:], _chk.stderr[-3000:], sep="\n")
    if "TorchAudio" in _chk.stderr: print("→ torchaudio가 아직 남아 있습니다. 새 셀에서 !pip uninstall -y torchaudio 를 한 번 더 실행한 뒤 이 셀을 다시 실행하세요")
    raise RuntimeError("설치 점검 실패 (위 오류) → 위 출력을 확인하세요")
print("✅ 설치 점검 OK (torch · CUDA · torchvision · transformers · vllm):", _line[-1][6:])
os.makedirs(HF_CACHE, exist_ok=True)
_free = shutil.disk_usage(HF_CACHE).free / 1e9
print(f"볼륨 디스크(/workspace) 여유 {_free:.0f}GB — 모델 크기만큼 필요 (7-0 셀에서 확인)")
if shutil.which("7z") is None:
    subprocess.run("apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq p7zip-full unzip", shell=True, capture_output=True, text=True)
print("압축 도구:", shutil.which("7z") or shutil.which("unzip") or "없음 → 3번 셀이 파이썬 zipfile로 풉니다(느림)")


## 3. 대회 데이터 (HF 비공개 저장소의 zip → /workspace/data)


In [ ]:
import os, subprocess, shutil, getpass, time
from huggingface_hub import login, whoami, hf_hub_download
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN", "")
if not HF_TOKEN: HF_TOKEN = getpass.getpass("Hugging Face 토큰(Write 권한, hf_로 시작)을 붙여 넣고 Enter: ").strip()
login(token=HF_TOKEN, add_to_git_credential=False)
HF_USER = whoami(token=HF_TOKEN)["name"]
HF_REPO = HF_REPO or f"{HF_USER}/ssafy-vqa-private"
print(f"Hugging Face 로그인: {HF_USER} | 비공개 저장소: {HF_REPO}")
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
def _find_data_dir(root):
    if os.path.exists(os.path.join(root, "train.csv")): return root
    for d, _, fs in os.walk(root):
        if "train.csv" in fs and "test.csv" in fs: return d
    return None
if _find_data_dir(DATA_DIR) is None:
    if not os.path.exists(ZIP_PATH):
        t0 = time.time(); print(f"대회 데이터 받는 중: {HF_REPO}/{HF_ZIP_NAME}")
        try:
            _got = hf_hub_download(HF_REPO, HF_ZIP_NAME, repo_type="dataset", token=HF_TOKEN, local_dir=os.path.dirname(ZIP_PATH))
        except Exception as e:
            raise SystemExit(f"❌ {HF_REPO}에서 {HF_ZIP_NAME}를 받지 못했습니다 ({type(e).__name__}: {str(e)[:200]})\n"
                             "→ Colab 브리지 노트북 A 단계를 먼저 실행했는지 확인하세요. 또는 Jupyter 왼쪽 파일 창에 zip을 /workspace로 직접 올리세요.")
        if os.path.abspath(_got) != os.path.abspath(ZIP_PATH): shutil.move(_got, ZIP_PATH)
        print(f"  받음 {os.path.getsize(ZIP_PATH) / 1e9:.2f}GB ({time.time() - t0:.0f}초)")
    t0 = time.time(); print("압축 해제 중... (몇 분)")
    if shutil.which("7z") or shutil.which("unzip"):
        _cmd = (["7z", "x", "-y", f"-p{ZIP_PASSWORD}", f"-o{DATA_DIR}", ZIP_PATH] if shutil.which("7z")
                else ["unzip", "-q", "-o", "-P", ZIP_PASSWORD, ZIP_PATH, "-d", DATA_DIR])
        _r = subprocess.run(_cmd, capture_output=True, text=True)
        assert _r.returncode == 0, f"압축 해제 실패 ({_cmd[0]}) → 비밀번호·파일을 확인하세요:\n" + (_r.stdout or "")[-600:] + (_r.stderr or "")[-800:]
    else:
        import zipfile
        with zipfile.ZipFile(ZIP_PATH) as _z: _z.extractall(DATA_DIR, pwd=ZIP_PASSWORD.encode())
    print(f"  완료 ({time.time() - t0:.0f}초)")
_dd = _find_data_dir(DATA_DIR)
assert _dd is not None, f"압축을 풀었는데 {DATA_DIR} 아래에 train.csv가 없습니다 → zip 내용을 확인하세요"
if _dd != DATA_DIR: print(f"train.csv가 하위 폴더에 있어 DATA_DIR을 바꿉니다: {_dd}"); DATA_DIR = _dd
for f in ["train.csv", "test.csv", "sample_submission.csv"]:
    assert os.path.exists(os.path.join(DATA_DIR, f)), f"{f} 가 {DATA_DIR} 에 없습니다"
print("데이터 OK:", DATA_DIR, "| 결과 저장:", OUT_DIR)


## 4. CSV · 팀 공통 검증 분할


In [ ]:
import re, json, time, math, hashlib, random, datetime, glob
import numpy as np, pandas as pd, torch
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None
random.seed(SEED); np.random.seed(SEED)

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sub_df   = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
dev_df   = pd.read_csv(os.path.join(DATA_DIR, "dev.csv")) if os.path.exists(os.path.join(DATA_DIR, "dev.csv")) else None
for c in ["question", "a", "b", "c", "d"]:
    train_df[c] = train_df[c].astype(str); test_df[c] = test_df[c].astype(str)
train_df["answer"] = train_df["answer"].astype(str).str.strip().str.lower()
assert set(train_df.answer.unique()) <= set("abcd"), train_df.answer.unique()
assert list(test_df.id) == list(sub_df.id), "test.csv 와 sample_submission.csv 의 id 순서가 다릅니다"
print(f"train {len(train_df):,} | test {len(test_df):,} | dev {0 if dev_df is None else len(dev_df):,}")
print("정답 분포:", train_df.answer.value_counts().sort_index().to_dict())

def img_path(rel):            # CSV의 path 컬럼은 DATA_DIR 기준 상대경로
    return os.path.join(DATA_DIR, rel)

EXPECTED_SPLIT_MD5 = "df05e4bc19d27ebceeef2570672149c2"

def make_split(df, val_size=VAL_SIZE, seed=SEED):
    rng = np.random.RandomState(seed)
    val_idx = []
    for letter in "abcd":
        idx = np.where(df.answer.values == letter)[0]
        rng.shuffle(idx)
        val_idx.extend(idx[: val_size // 4].tolist())
    val_idx = sorted(val_idx)
    mask = np.zeros(len(df), dtype=bool); mask[val_idx] = True
    return df[~mask].reset_index(drop=True), df[mask].reset_index(drop=True)

tr_df, val_df = make_split(train_df)
split_md5 = hashlib.md5(",".join(val_df.id.tolist()).encode()).hexdigest()
print(f"train {len(tr_df):,} / valid {len(val_df):,} | split md5 = {split_md5}")
print("valid 정답 분포:", val_df.answer.value_counts().sort_index().to_dict())
if split_md5 == EXPECTED_SPLIT_MD5:
    print("✅ 팀 공통 분할과 일치")
else:
    print("⚠️⚠️ 검증 분할이 기준과 다릅니다 → 팀원 간 점수 비교 불가. train.csv 원본/SEED/VAL_SIZE를 확인하세요.")
val_df[["id"]].to_csv(os.path.join(OUT_DIR, "valid_ids.csv"), index=False)

# E14 조합(검증 97.6%)의 오답 24개 — 라벨 오류 10 · 둘 다 답 8 · 진짜 실수 6 (OCR 실험 때와 같은 목록)
E14_ERR = {"라벨 오류": "0175 1878 3551 4755 6545 2114 4801 3823 6130 6211", "둘 다 답": "5008 1900 1911 2910 5811 2462 2186 1785",
           "진짜 실수": "2112 0936 5797 5209 1045 1000"}
E14_ERR = {k: [f"train_{x}.jpg" for x in v.split()] for k, v in E14_ERR.items()}

_rest = tr_df.sample(frac=1.0, random_state=VAL_ORDER_SEED)
VAL_ALL = pd.concat([val_df, _rest], ignore_index=True)
assert VAL_ALL.id.is_unique and len(VAL_ALL) == len(train_df)
VAL_USE = VAL_ALL.head(int(VAL_N)).copy()
FIXED_VAL_IDS = set(val_df.id)
print(f"검증 {len(VAL_USE):,}문항 사용 (팀 공통 {int(VAL_USE.id.isin(FIXED_VAL_IDS).sum()):,} + 나머지 {int((~VAL_USE.id.isin(FIXED_VAL_IDS)).sum()):,}) | test {len(test_df):,}")


## 5. 프롬프트 · 교차 검증 프롬프트 · 답 읽기


In [ ]:
PROMPT_VERSION = "baseline_v1"
SYSTEM_INSTRUCT = ("You are a helpful visual question answering assistant. "
                   "Answer using exactly one letter among a, b, c, or d. No explanation.")
LETTERS = "abcd"

def build_mc_prompt(question, options):
    a, b, c, d = options
    return (f"{question}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
            "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.")

def build_messages(row, perm=(0, 1, 2, 3)):
    # perm[k] = 화면의 k번째 자리에 놓을 '원래 선지'의 인덱스
    opts = [str(row[LETTERS[j]]) for j in perm]
    user_text = build_mc_prompt(str(row["question"]), opts)
    if globals().get("USE_SYSTEM_ROLE", True):
        return [{"role": "system", "content": SYSTEM_INSTRUCT if globals().get("SYSTEM_AS_STRING") else [{"type": "text", "text": SYSTEM_INSTRUCT}]},
                {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_text}]}]
    # system 역할이 없는 템플릿(일부 Gemma 등): 지시문을 user 메시지 앞에 붙임
    return [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": SYSTEM_INSTRUCT + "\n\n" + user_text}]}]

def load_image(rel):
    im = Image.open(img_path(rel))
    im = ImageOps.exif_transpose(im)      # 촬영 방향(EXIF) 보정 — 없으면 그대로
    return im.convert("RGB")
if PROMPT_VERSION_USE == "ko_v1":
    PROMPT_VERSION = "ko_v1"
    SYSTEM_INSTRUCT = "당신은 이미지를 보고 객관식 질문에 답하는 도우미입니다. a, b, c, d 중 정확히 한 글자로만 답하고 설명하지 마세요."
assert PROMPT_VERSION == PROMPT_VERSION_USE, (PROMPT_VERSION, PROMPT_VERSION_USE)

from transformers import AutoProcessor
import transformers
HCX_REMOTE_REV = "a6cdfd3464d1b767259cad23e164eaf39d3e3960"   # HyperCLOVAX-SEED-Think-32B 원격 코드판 (vLLM 0.30 시험 기준 커밋)
_IS_HCX = "hyperclova" in MODEL_ID.lower()
def _load_processor(rev):
    p = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=TRUST_REMOTE_CODE, revision=rev)
    if _IS_HCX: assert hasattr(p, "image_processor"), f"이미지 처리기가 없는 프로세서({type(p).__name__})가 불러와짐"
    return p
try:
    processor = _load_processor(REVISION)
except Exception as _e:
    if not (_IS_HCX and REVISION is None): raise
    # MODEL_PRESETS의 고정을 지운 경우: main은 transformers 네이티브 형식이라 실패 → 원격 코드판 커밋으로 고정 (8·9번 셀도 같은 커밋)
    print(f"⚠️ {MODEL_ID} main 프로세서 로드 실패 ({type(_e).__name__}: {str(_e)[:160]}) → 원격 코드판 커밋 {HCX_REMOTE_REV[:7]}로 고정해 다시 시도")
    REVISION = HCX_REMOTE_REV
    processor = _load_processor(REVISION)
tok = getattr(processor, "tokenizer", processor)
_tmpl = getattr(processor, "chat_template", None) or getattr(tok, "chat_template", None)
_tmpl = _tmpl if isinstance(_tmpl, str) else json.dumps(_tmpl)
def _think_kw(on):
    # 생각 모드를 켜고 끄는 템플릿 변수: Qwen·Gemma는 enable_thinking, HyperCLOVA X는 skip_reasoning (또는 thinking). 템플릿에 있는 것만 넘김
    if "enable_thinking" in (_tmpl or ""): return {"enable_thinking": bool(on)}
    kw = {}
    if re.search(r"\bthinking\b", _tmpl or ""): kw["thinking"] = bool(on)
    if "skip_reasoning" in (_tmpl or ""): kw["skip_reasoning"] = not on
    return kw
def _close_think(t, on):
    # 생각을 끄는데도 프롬프트가 '<think>'로 열린 채 끝나면 빈 생각 블록을 닫음 → 모델이 바로 답(풀이 + 정답 줄)을 씀
    if on: return t
    if t.endswith("<think>\n"): return t + "\n</think>\n\n"
    if t.endswith("<think>"): return t + "\n\n</think>\n\n"
    return t
TEMPLATE_KW = _think_kw(False)
_apply = processor.apply_chat_template if hasattr(processor, "apply_chat_template") and getattr(processor, "chat_template", None) else tok.apply_chat_template
try:
    _apply(build_messages({"question": "q", "a": "1", "b": "2", "c": "3", "d": "4"}), tokenize=False, add_generation_prompt=True, **TEMPLATE_KW); USE_SYSTEM_ROLE = True
except Exception as _e:
    USE_SYSTEM_ROLE = False; print("system 역할 미지원 템플릿 → 지시문을 user 메시지 앞에 붙입니다")
_raw_list = lambda t: "'type': 'text'" in t or '"type": "text"' in t      # 목록형 content가 풀리지 않고 글자 그대로 찍혔는지
SYSTEM_AS_STRING = False
if USE_SYSTEM_ROLE and _raw_list(_apply(build_messages({"question": "q", "a": "1", "b": "2", "c": "3", "d": "4"}), tokenize=False, add_generation_prompt=True, **TEMPLATE_KW)):
    SYSTEM_AS_STRING = True; print("템플릿이 목록형 system 내용을 풀지 못함 → system 지시문을 문자열로 넣습니다")
IMAGE_TOKEN = getattr(processor, "image_token", None)
IMAGE_TOKEN_ID = getattr(processor, "image_token_id", None)
if IMAGE_TOKEN_ID is None and IMAGE_TOKEN: IMAGE_TOKEN_ID = tok.convert_tokens_to_ids(IMAGE_TOKEN)
PROC_NAME = type(processor).__name__
FAMILY = "gemma4" if PROC_NAME.startswith("Gemma4") else ("qwen" if "Qwen" in PROC_NAME or hasattr(getattr(processor, "image_processor", None), "merge_size") else "other")

def render(row):
    return _close_think(_apply(build_messages(row), tokenize=False, add_generation_prompt=True, **TEMPLATE_KW), False) + ANSWER_PREFIX

_p = render(VAL_USE.iloc[0].to_dict())
if IMAGE_TOKEN and _p.count(IMAGE_TOKEN) != 1:
    print(f"ℹ️ 렌더링한 프롬프트에 이미지 토큰 {IMAGE_TOKEN!r}가 {_p.count(IMAGE_TOKEN)}번 보입니다 (템플릿이 다른 자리표시를 씀 → vLLM이 처리. 첫 묶음 점검에서 확인)")
print(f"transformers {transformers.__version__} | {PROC_NAME} (계열 {FAMILY}) | 템플릿 추가 인자 {TEMPLATE_KW} | system 역할 {USE_SYSTEM_ROLE} | 이미지 토큰 {IMAGE_TOKEN}={IMAGE_TOKEN_ID}")
print("프롬프트 끝부분:", repr(_p[-160:]))

LETTER_IDS = []                                     # 첫 번째(소문자·공백 없음)가 학습 목표, 나머지 변형은 확률 집계용
for L in LETTERS:
    ids = []
    for s in (L, L.upper(), " " + L, " " + L.upper()):
        t = tok.encode(s, add_special_tokens=False)
        if len(t) == 1 and t[0] not in ids: ids.append(t[0])
    assert ids and len(tok.encode(L, add_special_tokens=False)) == 1, f"'{L}' 가 단일 토큰이 아닙니다"
    LETTER_IDS.append(ids)
print("글자 토큰 id:", dict(zip(LETTERS, LETTER_IDS)))

# ── 교차 검증 프롬프트 (E20과 같은 문구, 결과를 보기 전에 고정) ──
XC_TEMPLATE_KW = _think_kw(THINKING)
XC_SYSTEM = "당신은 이미지 속 글자와 내용을 꼼꼼히 확인해 객관식 문제의 정답을 판정하는 검증자입니다."
def xc_user_text(row, A, B):
    opts = "\n".join(f"({L}) {row[L]}" for L in LETTERS)
    return (f"{row['question']}\n{opts}\n\n"
            "[작전 지시: 오답 돌파 프로토콜]\n"
            "1. 두 모델의 판단이 갈렸다.\n"
            f"   - 모델 A의 답: ({LETTERS[A]}) {row[LETTERS[A]]}\n"
            f"   - 모델 B의 답: ({LETTERS[B]}) {row[LETTERS[B]]}\n"
            "2. 이미지에서 질문과 관련된 부분을 찾아 글자와 숫자를 정확히 읽고, 각 모델이 전제로 삼은 근거에서 논리적 모순이나 비약이 있는지 교차 검증하라.\n"
            "3. 이 문제 유형에서 자주 발생하는 대표적 착각 패턴을 배제하고, 최종 결론을 단계별로 도출하라.\n"
            "마지막 줄에는 반드시 \"정답: (a)\"처럼 a, b, c, d 중 하나만 적어라.")
def xc_messages(row, A, B):
    t = xc_user_text(row, A, B)
    if USE_SYSTEM_ROLE:
        return [{"role": "system", "content": XC_SYSTEM if SYSTEM_AS_STRING else [{"type": "text", "text": XC_SYSTEM}]},
                {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": t}]}]
    return [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": XC_SYSTEM + "\n\n" + t}]}]
def xc_pair(qid, anchor, alt):
    # 기준 답·대안을 모델 A/B 자리에 무작위로 (문항 id로 고정 → 다시 돌려도 같음, 판정 모델이 어느 쪽이 기준인지 모름)
    flip = int(hashlib.md5(f"{qid}|xc".encode()).hexdigest(), 16) % 2
    return (alt, anchor) if flip else (anchor, alt)
def render_xc(row, A, B):
    return _close_think(_apply(xc_messages(row, A, B), tokenize=False, add_generation_prompt=True, **XC_TEMPLATE_KW), THINKING)
# 마지막 '정답: (x)'를 읽음. 괄호 없는 대문자 A·B는 '모델 A/B'와 헷갈려서 답으로 치지 않음
_ANS_RE = re.compile(r"정답\s*[:：]?\s*\**\s*[\(\[（]\s*([abcdABCD])\s*[\)\]）]|정답\s*[:：]\s*\**\s*([abcdCD])(?![A-Za-z가-힣])")
def parse_answer(text):
    t = text.split("</think>")[-1]                         # 생각 모드면 생각 뒤 부분만
    m = list(_ANS_RE.finditer(t))
    if m: return (m[-1].group(1) or m[-1].group(2)).lower()
    tail = t.strip().splitlines()[-1] if t.strip() else ""
    m2 = re.findall(r"\(([abcd])\)", tail)
    return m2[-1] if m2 else None
for _s, _e in [("분석 끝.\n정답: (c)", "c"), ("정답은 (a)일 수도 있지만 ... \n정답: b", "b"), ("최종 정답：（d）", "d"), ("답을 모르겠다", None),
               ("<think>정답: (a)</think>\n정답: (c)", "c"), ("**정답: (b)**", "b"), ("정답: 모델 A", None), ("정답: B", None), ("정답: d", "d")]:
    assert parse_answer(_s) == _e, (_s, parse_answer(_s))
_r0 = VAL_USE.iloc[0].to_dict(); _A0, _B0 = xc_pair(_r0["id"], 0, 1)
_px = render_xc(_r0, _A0, _B0)
if IMAGE_TOKEN and _px.count(IMAGE_TOKEN) != 1: print(f"ℹ️ 판정 프롬프트에 이미지 토큰 {IMAGE_TOKEN!r}가 {_px.count(IMAGE_TOKEN)}번 보입니다 (첫 묶음 점검에서 확인)")
print(f"판정 템플릿 추가 인자 {XC_TEMPLATE_KW} | system 문자열 {SYSTEM_AS_STRING} | 저장소 커밋 {REVISION or 'main'} | 예시 끝부분:"); print(_px[-600:])
assert not _raw_list(_px), f"판정 프롬프트에 목록형 content가 글자 그대로 찍힘 → 템플릿이 예상과 다름: {_px[-400:]!r}"
if _IS_HCX:                                  # vLLM 전용 구현은 <|IMAGE_PAD|> 한 개를 이미지 토큰들로 바꿈 · 생각 블록이 닫혀 있어야 바로 풀이+정답을 씀
    assert _px.count("<|IMAGE_PAD|>") == 1, f"HyperCLOVA X 판정 프롬프트에 이미지 자리(<|IMAGE_PAD|>)가 {_px.count('<|IMAGE_PAD|>')}번 → 템플릿이 예상과 다름: {_px[-400:]!r}"
    assert THINKING or _px.endswith("</think>\n\n"), f"HyperCLOVA X 생각 블록이 닫히지 않음: {_px[-120:]!r}"
    print(f"✅ HyperCLOVA X: 커밋 {(REVISION or 'main')[:7]} · 이미지 자리 1개 · 생각 블록 닫힘 (바로 풀이 + '정답: (x)')")


## 6. 판정 목록 받기 (Colab `anchor_rescue` 6-1번 셀이 만든 것)
- 모델을 받기 전에 대상 문항 수를 먼저 보여 줍니다.


In [ ]:
from huggingface_hub import hf_hub_download
if TARGETS_FILE:
    _tp = TARGETS_FILE
else:
    _tp = hf_hub_download(HF_REPO, "xcheck/xcheck_targets.csv", repo_type="dataset", token=HF_TOKEN or None, force_download=True, local_dir="/workspace/xcheck")
assert os.path.exists(_tp), f"판정 목록이 없습니다: {_tp}"
TG = pd.read_csv(_tp)
_need = {"id", "split", "anchor", "alt", "n_alt", "target", "list_tag", "anchor_run"}
assert _need <= set(TG.columns), f"판정 목록 열이 부족합니다: {list(TG.columns)} → Colab anchor_rescue 6-1번 셀이 만든 파일인지 확인하세요"
TG_TAG, XC_ANCHOR = str(TG.list_tag.iloc[0]), str(TG.anchor_run.iloc[0])
_tv, _tt = TG[TG.split == "val"].set_index("id"), TG[TG.split == "test"].set_index("id")
assert set(_tv.index) == set(val_df.id), "판정 목록의 검증 문항이 팀 공통 분할과 다릅니다 → 위 출력을 확인하세요"
assert set(_tt.index) == set(test_df.id), "판정 목록의 test 문항이 test.csv와 다릅니다 → 위 출력을 확인하세요"
assert TG.anchor.isin(list(LETTERS)).all() and TG.alt.isin(list(LETTERS)).all() and (TG[TG.target == 1].anchor != TG[TG.target == 1].alt).all(), "판정 목록의 답 글자 이상"
ANCH_V, ANCH_T = _tv.anchor.to_dict(), _tt.anchor.to_dict()
XV = val_df.merge(_tv[_tv.target == 1][["anchor", "alt", "n_alt"]].reset_index(), on="id")       # 검증 대상 (정답 있음 → 판정 모델 성적)
XT = test_df.merge(_tt[_tt.target == 1][["anchor", "alt", "n_alt"]].reset_index(), on="id")
_aw = int((_tv.loc[val_df.id].anchor.to_numpy() != val_df.answer.to_numpy()).sum())
print(f"판정 목록 {TG_TAG} (기준 {XC_ANCHOR}) | 검증 대상 {len(XV)}/{len(val_df)} — 기준 오답 {int((XV.anchor != XV.answer).sum())}/{_aw} 포함, 대안이 정답 {int((XV.alt == XV.answer).sum())}"
      f" | test 대상 {len(XT)}/{len(test_df)} | 표본 {K_SAMPLES}개씩 → 생성 {(len(XV) + len(XT)) * K_SAMPLES:,}개")


## 7. 이미지 해상도 점검 (zero-shot 노트북과 같음)


In [ ]:
if PRESET == "default" or FAMILY == "other":
    MM_KW = {}
elif FAMILY == "gemma4":
    MM_KW = {"max_soft_tokens": int(RES_PRESETS_GEMMA4[PRESET])}
else:
    MM_KW = {"min_pixels": int(RES_PRESETS[PRESET][0]), "max_pixels": int(RES_PRESETS[PRESET][1])}
if FAMILY == "other" and PRESET != "default":
    print(f"ℹ️ {PROC_NAME}은 해상도 인자를 모르는 계열이라 모델 기본 처리로 둡니다 (PRESET={PRESET!r}는 이름에만 남음)")
for _r in VAL_USE.head(2).to_dict("records"):
    _im = load_image(_r["path"])
    try:
        _enc = processor(text=[render(_r)], images=[[_im]] if FAMILY == "gemma4" else [_im], return_tensors="pt", **MM_KW)
        _n = int((_enc["input_ids"] == IMAGE_TOKEN_ID).sum()) if IMAGE_TOKEN_ID is not None else -1
        print(f"  {_r['id']}: 원본 {_im.size} → 프롬프트 {int(_enc['input_ids'].shape[1])}토큰 (이미지 토큰 {_n}) | 해상도 인자 {MM_KW or '기본'}")
    except Exception as _e:
        print(f"  ⚠️ HF 프로세서 점검 실패 ({type(_e).__name__}: {str(_e)[:160]}) → vLLM 쪽 처리로 진행 (첫 묶음 점검에서 확인)")


## 8. 모델 파일 받기 (이미 받은 부분은 다시 안 받음)


In [ ]:
import shutil
from huggingface_hub import snapshot_download, HfApi
try:
    _info = HfApi(token=HF_TOKEN).model_info(MODEL_ID, files_metadata=True, revision=REVISION)
    _need = sum((s.size or 0) for s in _info.siblings if s.rfilename.endswith(".safetensors")) / 1e9
    _free0 = shutil.disk_usage("/workspace").free / 1e9
    print(f"모델 가중치 {_need:.0f}GB | 볼륨 디스크 여유 {_free0:.0f}GB (이미 받은 부분은 다시 안 받음)")
    if _need > _free0 + 1: print("⚠️ 공간이 모자랄 수 있습니다 → Pods → ⋮ → Edit Pod에서 Volume Disk를 늘리세요")
except Exception as _e:
    print(f"(모델 크기 조회 실패: {type(_e).__name__} — 게이트 모델이면 HF에서 라이선스 동의가 필요합니다)")
t0 = time.time()
MODEL_DIR = snapshot_download(MODEL_ID, revision=REVISION, token=HF_TOKEN or None, max_workers=16, ignore_patterns=["*.pth", "*.pt", "*.gguf", "*.msgpack", "*.h5", "*.onnx", "original/*", "consolidated*"])
_sz = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(MODEL_DIR) for f in fs) / 1e9
_nsh = len([f for f in os.listdir(MODEL_DIR) if f.endswith(".safetensors")])
print(f"모델 파일 {_sz:.1f}GB · 가중치 조각 {_nsh}개 | {time.time() - t0:.0f}s | {MODEL_DIR}")
_free = shutil.disk_usage(HF_CACHE).free / 1e9 if os.path.exists(HF_CACHE) else float("nan")
print(f"로컬 디스크 여유 {_free:.0f}GB")


## 9. vLLM으로 모델 로드
오류가 나면 마지막 20줄을 확인하세요. 설정을 바꾼 뒤에는 **Kernel → Restart Kernel and Run All Cells**로 다시 시작하세요.


In [ ]:
import vllm, contextlib, io, sys
from vllm import LLM, SamplingParams
try:
    sys.stdout.fileno(); _FILENO_OK = True
except (io.UnsupportedOperation, AttributeError, ValueError):
    _FILENO_OK = False
if not _FILENO_OK:
    # Colab 노트북의 출력 스트림에는 파일 번호가 없음. vLLM은 분산 초기화 때 C 라이브러리 출력을 숨기려고 sys.stdout.fileno()를 부르다 멈춤
    # (엔진을 노트북 프로세스 안에서 돌리기 때문) → 그 숨기기 기능만 끔. 결과에는 영향 없고, gloo 초기화 메시지가 숨겨지지 않을 뿐
    import vllm.utils.system_utils as _su, vllm.distributed.parallel_state as _ps, vllm.distributed.utils as _du
    @contextlib.contextmanager
    def _no_suppress_stdout(): yield
    for _mod in (_su, _ps, _du): _mod.suppress_stdout = _no_suppress_stdout
    print("Colab 출력 스트림 → vLLM의 stdout 숨기기(suppress_stdout)를 끔")
if "llm" not in globals():
    t0 = time.time()
    llm = LLM(model=MODEL_ID, revision=REVISION, tokenizer_revision=REVISION, code_revision=REVISION, quantization=QUANTIZATION, dtype=DTYPE, max_model_len=MAX_MODEL_LEN, trust_remote_code=TRUST_REMOTE_CODE,
              gpu_memory_utilization=GPU_MEM_UTIL, max_num_seqs=MAX_NUM_SEQS, max_num_batched_tokens=MAX_BATCHED_TOKENS,
              limit_mm_per_prompt={"image": 1, "video": 0}, mm_processor_kwargs=MM_KW, mm_processor_cache_gb=0,
              enforce_eager=True, enable_prefix_caching=False, max_logprobs=TOPK_LOGPROBS, seed=SEED, moe_backend=MOE_BACKEND,
              gdn_prefill_backend=GDN_PREFILL_BACKEND, **ENGINE_KW)
    print(f"모델 로드 {time.time() - t0:.0f}s")
else:
    print("이미 로드된 모델을 그대로 씀 (설정을 바꿨다면 세션을 다시 시작하세요)")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
_free, _tot = torch.cuda.mem_get_info() if torch.cuda.is_available() else (0, 0)
print(f"vllm {vllm.__version__} | {MODEL_ID} | {GPU_NAME} | GPU 사용 {(_tot - _free) / 2**30:.1f} / {_tot / 2**30:.0f}GiB")
SP = SamplingParams(max_tokens=1, temperature=0.0, logprobs=TOPK_LOGPROBS, detokenize=False)


## 10. 판정 생성 함수 (끊겨도 이어서)
- 문항마다 표본 `K_SAMPLES`개를 한 요청(`n`)으로 뽑습니다. 문항별 시드를 써서 다시 돌려도 같습니다.
- 표본마다 `id, split, sample, letter, finish, ntok, A, B, text`를 JSONL로 저장합니다. 목록이 바뀌어 A/B 짝이 달라진 문항은 새로 판정합니다.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
RUN_DIR = os.path.join(OUT_DIR, RUN_NAME); os.makedirs(RUN_DIR, exist_ok=True)
GEN_VAL, GEN_TEST = os.path.join(RUN_DIR, f"xc_val__{PRESET}.jsonl"), os.path.join(RUN_DIR, f"xc_test__{PRESET}.jsonl")
_XPOOL, _XPRE = ThreadPoolExecutor(max_workers=IMG_WORKERS), ThreadPoolExecutor(max_workers=1)
GEN_COLS = ["id", "split", "sample", "letter", "finish", "ntok", "A", "B", "text"]
def _seed_of(qid): return int(hashlib.md5(f"{qid}|{SEED}".encode()).hexdigest(), 16) % (2 ** 31)
def _xc_req(item):
    r, A, B = item
    return {"prompt": render_xc(r, A, B), "multi_modal_data": {"image": load_image(r["path"])}}
def read_gen(path):
    rows = []
    if os.path.exists(path):
        for line in open(path, encoding="utf-8"):
            try: rows.append(json.loads(line))
            except Exception: pass                         # 끊겨서 반쯤 쓴 줄
    return pd.DataFrame(rows, columns=GEN_COLS)
def _fix_tail(path):                   # 저장 도중 끊겨 마지막 줄이 줄바꿈 없이 끝났으면 줄바꿈을 붙여 다음 줄과 섞이지 않게
    if os.path.exists(path) and os.path.getsize(path) > 0:
        with open(path, "rb+") as f:
            f.seek(-1, os.SEEK_END)
            if f.read(1) != b"\n": f.write(b"\n")
def _items(df):
    return [(r, *xc_pair(r["id"], LETTERS.index(r["anchor"]), LETTERS.index(r["alt"]))) for r in df.to_dict("records")]
def _done_counts(path):
    # 같은 문항·같은 A/B 짝으로 이미 뽑은 표본 수 (목록이 바뀌어 짝이 달라진 문항은 새로 판정)
    d = read_gen(path)
    if not len(d): return {}
    return d.drop_duplicates(["id", "A", "B", "sample"], keep="last").groupby(["id", "A", "B"]).size().to_dict()
def generate_xc(df, path, split):
    _fix_tail(path)
    cnt = _done_counts(path)
    items = [(r, A, B) for r, A, B in _items(df) if cnt.get((r["id"], LETTERS[A], LETTERS[B]), 0) < K_SAMPLES]
    print(f"[{split}] {os.path.basename(path)} | 대상 {len(df)}문항 | 완료 {len(df) - len(items)} | 남음 {len(items)} | 표본 {K_SAMPLES}개씩", flush=True)
    if not items: return read_gen(path)
    chunks = [items[i:i + GEN_CHUNK] for i in range(0, len(items), GEN_CHUNK)]
    fut, t0, n, ntok = _XPRE.submit(lambda c: list(_XPOOL.map(_xc_req, c)), chunks[0]), time.time(), 0, 0
    for k, ch in enumerate(chunks):
        reqs = fut.result()
        if k + 1 < len(chunks): fut = _XPRE.submit(lambda c: list(_XPOOL.map(_xc_req, c)), chunks[k + 1])
        sps = [SamplingParams(n=K_SAMPLES, temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K, max_tokens=MAX_TOKENS, seed=_seed_of(r["id"])) for r, _, _ in ch]
        outs = llm.generate(reqs, sps, use_tqdm=False)
        if len(outs) > len(ch):
            print(f"  (엔진에 남아 있던 이전 요청 {len(outs) - len(ch)}개의 결과는 버림)"); outs = outs[-len(ch):]
        assert len(outs) == len(ch) and all(getattr(o, "prompt", None) in (None, q["prompt"]) for o, q in zip(outs, reqs)), "요청과 결과의 순서가 맞지 않음"
        with open(path, "a", encoding="utf-8") as f:
            for (r, A, B), o in zip(ch, outs):
                have = int(cnt.get((r["id"], LETTERS[A], LETTERS[B]), 0))
                for j, c in enumerate(o.outputs[have:K_SAMPLES], start=have):
                    f.write(json.dumps({"id": r["id"], "split": split, "sample": j, "letter": parse_answer(c.text), "finish": c.finish_reason,
                                        "ntok": len(c.token_ids), "A": LETTERS[A], "B": LETTERS[B], "text": c.text}, ensure_ascii=False) + "\n")
                    ntok += len(c.token_ids)
        n += len(ch)
        if k == 0:
            _d = read_gen(path); _d = _d[_d.id.isin([r["id"] for r, _, _ in ch])]
            print(f"  첫 묶음 점검: 답을 읽은 표본 {int(_d.letter.notna().sum())}/{len(_d)} | 길이 평균 {_d.ntok.mean():.0f}토큰 (최대 {_d.ntok.max()}) | 길이 제한에 걸림 {int((_d.finish == 'length').sum())}")
            print("  예시 표본 끝부분:", repr(_d.text.iloc[0][-300:]))
            if _d.letter.notna().mean() < 0.5:
                raise RuntimeError("답을 읽어 낸 표본이 절반도 안 됩니다 → 마지막 줄 형식을 안 지키거나 길이 제한에 걸림. "
                                   "'길이 제한에 걸림'이 많으면 1번 셀 MAX_TOKENS를 늘리고, 아니면 위 예시로 답 형식을 확인하세요")
        if k == 0 or (k + 1) % 5 == 0 or k + 1 == len(chunks):
            el = time.time() - t0
            print(f"  {n:>4}/{len(items)}문항 | {el / n:.1f}s/문항 | 생성 {ntok / max(el, 1e-9):.0f}토큰/s | 남은 시간 ~{(len(items) - n) * el / n / 60:.0f}분", flush=True)
    return read_gen(path)
def counts_of(gen, df):
    # 문항별 표본 답 수 (지금 목록의 A/B 짝으로 뽑은 표본만)
    want = {r["id"]: (LETTERS[A], LETTERS[B]) for r, A, B in _items(df)}; out = {}
    if not len(gen): return out
    g = gen.drop_duplicates(["id", "A", "B", "sample"], keep="last")
    for (qid, a_, b_), sub in g.groupby(["id", "A", "B"]):
        if want.get(qid) != (a_, b_): continue
        v = sub.letter.dropna(); v = v[v.isin(list(LETTERS))]
        out[qid] = np.bincount(v.map(LETTERS.index).astype(int).to_numpy(), minlength=4)
    return out
def decide(c):
    # 이 판정 모델의 한 표: 읽은 표본 ≥ MIN_VALID이고 1등이 하나일 때만. 아니면 기권(None)
    if c is None or c.sum() < MIN_VALID: return None
    w = np.flatnonzero(c == c.max()); return int(w[0]) if len(w) == 1 else None
def preds_frame(ids, anchor_map, cnt):
    # E14 형식: 판정한 문항은 표본 비율, 기권·목록 밖 문항은 기준 답(= 다른 보기에 표를 주지 않음)
    rows = []
    for i in ids:
        a = LETTERS.index(anchor_map[i]); c = cnt.get(i); v = decide(c)
        p = (c / c.sum()) if v is not None else np.eye(4)[a]
        rows.append({"id": i, "pred": LETTERS[int(p.argmax())], "p_a": p[0], "p_b": p[1], "p_c": p[2], "p_d": p[3], "letter_mass": 1.0,
                     "judged": int(v is not None), "vote": LETTERS[v] if v is not None else "", "n_valid": int(c.sum()) if c is not None else 0, "anchor": anchor_map[i]})
    return pd.DataFrame(rows)
SUMMARY = {"lines": []}
def say(s): print(s); SUMMARY["lines"].append(s)
print("✅ 생성 함수 준비 완료 | 저장:", RUN_DIR)


## 11. 판정 실행: 검증 대상 → test 대상
- 검증 대상에서는 정답을 알기 때문에 이 판정 모델 혼자의 성적(판정 다수결 vs 기준)을 보여 줍니다. 참고용이고, 이것으로 판정 모델을 고르지 않습니다.
- "A 자리 / B 자리" 수가 한쪽으로 크게 쏠리면 보기 자리 편향입니다.


In [ ]:
GEN_V = generate_xc(XV, GEN_VAL, "val"); CNT_V = counts_of(GEN_V, XV)
GEN_T = generate_xc(XT, GEN_TEST, "test"); CNT_T = counts_of(GEN_T, XT)
_rows = []
for r, A, B in _items(XV):
    c = CNT_V.get(r["id"]); v = decide(c)
    _rows.append({"id": r["id"], "정답": r["answer"], "기준": r["anchor"], "대안": r["alt"], "판정": LETTERS[v] if v is not None else "기권",
                  "A자리": LETTERS[A], "B자리": LETTERS[B], "투표": " ".join(f"{L}{int(c[k])}" for k, L in enumerate(LETTERS) if c[k] > 0) if c is not None else "", "대안 표": r["n_alt"],
                  "질문": str(r["question"])[:34]})
JV = pd.DataFrame(_rows, columns=["id", "정답", "기준", "대안", "판정", "A자리", "B자리", "투표", "대안 표", "질문"]); dec = JV[JV.판정 != "기권"]
_g = pd.concat([GEN_V, GEN_T]) if len(GEN_T) else GEN_V
say(f"[판정 {MODEL_ID}] 검증 대상 {len(JV)} | 판정 {len(dec)} (기권 {len(JV) - len(dec)}) | 판정 정답 {int((dec.판정 == dec.정답).sum())} vs 같은 문항 기준 정답 {int((dec.기준 == dec.정답).sum())}"
    f" | 판정 = 기준 {int((dec.판정 == dec.기준).sum())} · = 대안 {int((dec.판정 == dec.대안).sum())} · 그 밖 {int(((dec.판정 != dec.기준) & (dec.판정 != dec.대안)).sum())}")
_ch = dec[dec.판정 != dec.기준]
say(f"  기준과 다른 답 {len(_ch)}문항: 맞게 바꿈 {int((_ch.판정 == _ch.정답).sum())} · 기준이 맞았는데 바꿈 {int((_ch.기준 == _ch.정답).sum())} (이 모델 혼자 기준. 실제로 뒤집을지는 anchor_rescue가 정함)")
say(f"  표본 {len(_g)}개: 답 읽음 {int(_g.letter.notna().sum())} ({_g.letter.notna().mean() * 100:.1f}%) · 길이 제한 {int((_g.finish == 'length').sum())} · 평균 {_g.ntok.mean():.0f}토큰"
    f" | 판정이 A 자리 보기 {int((dec.판정 == dec.A자리).sum())} · B 자리 {int((dec.판정 == dec.B자리).sum())} · 둘 다 아님 {int(((dec.판정 != dec.A자리) & (dec.판정 != dec.B자리)).sum())} (A·B가 한쪽으로 크게 쏠리면 자리 편향)")
if len(_ch): print("\n기준과 다른 답을 고른 검증 문항:"); print(_ch.drop(columns=["A자리", "B자리"]).to_string(index=False))
_dt = [decide(CNT_T.get(i)) for i in XT.id]
say(f"  test 대상 {len(XT)} | 판정 {sum(v is not None for v in _dt)} · 기준과 다른 답 {sum(v is not None and LETTERS[v] != a for v, a in zip(_dt, XT.anchor))}")


## 12. E14 형식 저장 (anchor_rescue가 한 표로 읽는 파일) · 재현 점검


In [ ]:
VAL_FINAL = os.path.join(RUN_DIR, f"preds_final_ep1__{PRESET}__{len(val_df)}.csv")
TEST_FINAL = os.path.join(RUN_DIR, f"preds_final_ep1_test__{PRESET}__{len(test_df)}.csv")
PVX, PTX = preds_frame(val_df.id.tolist(), ANCH_V, CNT_V), preds_frame(test_df.id.tolist(), ANCH_T, CNT_T)
PVX.to_csv(VAL_FINAL, index=False); PTX.to_csv(TEST_FINAL, index=False)
acc_x = float((PVX.pred.to_numpy() == val_df.answer.to_numpy()).mean())
json.dump({"epoch_acc": {"1": acc_x},
           "config": {"MODEL_ID": "xcheck:" + MODEL_ID, "JUDGE_MODEL_ID": MODEL_ID, "ZERO_SHOT": True, "XCHECK": True, "XCHECK_ANCHOR": XC_ANCHOR, "TARGETS_TAG": TG_TAG,
                      "TRAIN_ON_FULL": False, "SEED": SEED, "EVAL_PRESET": PRESET, "PROMPT_VERSION": "xcheck_v1", "K_SAMPLES": K_SAMPLES, "TEMPERATURE": TEMPERATURE,
                      "TOP_P": TOP_P, "TOP_K": TOP_K, "MAX_TOKENS": MAX_TOKENS, "MIN_VALID": MIN_VALID, "THINKING": THINKING, "TEMPLATE_KW": XC_TEMPLATE_KW, "REVISION": REVISION, "SYSTEM_AS_STRING": SYSTEM_AS_STRING,
                      "MM_KW": MM_KW, "ENGINE": f"vllm {vllm.__version__}", "QUANTIZATION": QUANTIZATION, "SPLIT_MD5": split_md5},
           "xcheck": {"val_targets": len(XV), "val_judged": int(PVX.judged.sum()), "test_targets": len(XT), "test_judged": int(PTX.judged.sum()), "lines": SUMMARY["lines"]}},
          open(os.path.join(RUN_DIR, "train_log.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=str)
# 재현 점검: 저장한 생성 파일만으로 다시 만들어 같은 표인지
_rv = preds_frame(val_df.id.tolist(), ANCH_V, counts_of(read_gen(GEN_VAL), XV)); _rt = preds_frame(test_df.id.tolist(), ANCH_T, counts_of(read_gen(GEN_TEST), XT))
_same = bool((_rv.pred.to_numpy() == PVX.pred.to_numpy()).all() and (_rt.pred.to_numpy() == PTX.pred.to_numpy()).all())
say(f"✅ E14 형식 저장: {os.path.basename(VAL_FINAL)} (판정 {int(PVX.judged.sum())}) · {os.path.basename(TEST_FINAL)} (판정 {int(PTX.judged.sum())}) · train_log.json | 재현 {'✅' if _same else '⚠️'} → {RUN_DIR}")
assert _same, "저장한 생성 파일로 다시 만든 표가 다릅니다 → 위 출력을 확인하세요"


## 13. 결과를 HF 비공개 저장소로 올리기 → Colab 브리지 B 단계


In [ ]:
import re, time
from huggingface_hub import HfApi
_fs = os.listdir(RUN_DIR)
_val = sorted(f for f in _fs if re.match(r"^preds_final_ep\d+__.+__" + str(len(val_df)) + r"\.csv$", f))
_tst = sorted(f for f in _fs if re.match(r"^preds_final_ep\d+_test__.+__" + str(len(test_df)) + r"\.csv$", f))
assert _val and _tst, f"검증 1,000 · test 예측 파일이 없습니다 ({RUN_DIR}) → 11·12번 셀이 끝났는지 확인하세요"
for f, n in [(x, len(val_df)) for x in _val] + [(x, len(test_df)) for x in _tst]:
    _d = pd.read_csv(os.path.join(RUN_DIR, f))
    assert _d.id.nunique() == n and _d[["p_a", "p_b", "p_c", "p_d"]].notna().all().all(), f"{f}: 문항 {_d.id.nunique()}/{n} 또는 확률 열 이상"
api = HfApi(token=HF_TOKEN)
api.create_repo(HF_REPO, repo_type="dataset", private=True, exist_ok=True)
assert api.repo_info(HF_REPO, repo_type="dataset").private, f"{HF_REPO}가 공개 저장소입니다 → HF 웹에서 Private로 바꾼 뒤 다시 실행하세요"
t0 = time.time(); _pre = f"vqa_runs/{RUN_NAME}"
print(f"올리는 중: {RUN_DIR} → {HF_REPO}/{_pre}")
api.upload_folder(repo_id=HF_REPO, repo_type="dataset", folder_path=RUN_DIR, path_in_repo=_pre, commit_message=f"RunPod xcheck {RUN_NAME}")
_remote = set(api.list_repo_files(HF_REPO, repo_type="dataset"))
_miss = [f for f in [f"{_pre}/{x}" for x in _val + _tst + ["train_log.json"]] if f not in _remote]
assert not _miss, f"올라가지 않은 파일: {_miss}"
print(f"✅ 올리기 완료 ({time.time() - t0:.0f}초) | {_val} · {_tst} · train_log.json")
print("다음: ① 다른 판정 모델이면 MODEL_ID를 바꾸고 Kernel → Restart Kernel and Run All Cells ② 다 끝나면 Colab 브리지 노트북 B 단계 → anchor_rescue 다시 실행 → pod Terminate")


## 14. 결과 요약


In [ ]:
print(f"판정 | {MODEL_ID} | {RUN_NAME} | 목록 {TG_TAG} (기준 {XC_ANCHOR}) | 표본 {K_SAMPLES} · T {TEMPERATURE} · 최대 {MAX_TOKENS}토큰 · 생각 {THINKING} | vllm {vllm.__version__} | {GPU_NAME}")
for _l in SUMMARY["lines"]: print(_l)
